In [1]:
import pandas as pd
import geopandas as gpd

gdf = gpd.read_file('../../Data_list/preprocessing_result/군집화포인트_등고선_인구병합결과/위치_등고선_시군구_비율.shp', encoding='cp949')
gdf.head()

,cluster_id,data_point,시군구,어린이비율,contour,contour_mi,contour_ma,high_up,high_down,geometry
0,0,1976,부산광역시 금정구,0.061106,40.0,15.0,130.0,25.0,90.0,POINT (129.1003 35.21535)
1,1,1883,부산광역시 해운대구,0.084278,55.0,20.0,210.0,35.0,155.0,POINT (129.15086 35.22258)
2,2,1435,부산광역시 금정구,0.061106,50.0,20.0,130.0,30.0,80.0,POINT (129.09771 35.21647)
3,3,1772,부산광역시 해운대구,0.084278,35.0,35.0,145.0,0.0,110.0,POINT (129.15593 35.23157)
4,4,2419,부산광역시 해운대구,0.084278,125.0,40.0,285.0,85.0,160.0,POINT (129.1311 35.19808)


In [2]:
candidate_gdf = pd.read_csv('../../Data_list/preprocessing_result/스코어링할데이터/candidate_gdf.csv')
candidate_gdf.head()

,cluster_id,data_point,geometry,residences_count,bus_count,train_count,parking_count,children_care_count,어린이,high_up,high_down
0,0,1976,POINT (129.10029965278912 35.21534903840932),13966,5,0,0,16,0.061106,25.0,90.0
1,1,1883,POINT (129.15085966138363 35.22258067813977),5294,6,0,0,7,0.084278,35.0,155.0
2,2,1435,POINT (129.09771423446185 35.21646805222987),13815,3,0,0,12,0.061106,30.0,80.0
3,3,1772,POINT (129.1559338144019 35.23156637192356),5347,5,0,1,12,0.084278,0.0,110.0
4,4,2419,POINT (129.131100786837 35.19807624497906),5430,4,0,1,12,0.084278,85.0,160.0


In [3]:
# 중복 컬럼을 제외한 candidate_gdf의 컬럼만 추출
duplicate_cols = set(gdf.columns) & set(candidate_gdf.columns)
candidate_gdf_unique = candidate_gdf[[col for col in candidate_gdf.columns if col not in duplicate_cols]]

# gdf와 candidate_gdf_unique를 컬럼 기준으로 합침 (index 맞추기)
merged_candidate_df = pd.concat([gdf.reset_index(drop=True), candidate_gdf_unique.reset_index(drop=True)], axis=1)

# 중복되는 데이터가 들어간 컬럼 제거
merged_candidate_df = merged_candidate_df.drop(columns=['어린이'])

In [4]:
# 일부 object 형태의 컬럼의 데이터타입 모두 숫자로 변경
merged_candidate_df[['contour_mi', 'contour_ma', 'high_up', 'high_down']] = merged_candidate_df[['contour_mi', 'contour_ma', 'high_up', 'high_down']].astype('float64')
merged_candidate_df.dtypes

cluster_id                int64
data_point                int64
시군구                      object
어린이비율                   float64
contour                 float64
contour_mi              float64
contour_ma              float64
high_up                 float64
high_down               float64
geometry               geometry
residences_count          int64
bus_count                 int64
train_count               int64
parking_count             int64
children_care_count       int64
dtype: object

In [5]:
merged_candidate_df.head(2)

,cluster_id,data_point,시군구,어린이비율,contour,contour_mi,contour_ma,high_up,high_down,geometry,residences_count,bus_count,train_count,parking_count,children_care_count
0,0,1976,부산광역시 금정구,0.061106,40.0,15.0,130.0,25.0,90.0,POINT (129.1003 35.21535),13966,5,0,0,16
1,1,1883,부산광역시 해운대구,0.084278,55.0,20.0,210.0,35.0,155.0,POINT (129.15086 35.22258),5294,6,0,0,7


In [ ]:
# 각 컬럼별 점수 계산 함수 정의

def score_bus_count(x):
    # 0~1: 0.5점, 2~3: 1점, 4~5: 1.5점, ... 18~19: 5점
    return 0.5 + (x // 2) * 0.5

def score_train_count(x):
    # 0: 0점, 1: 1점, 2: 2점
    return x

def score_parking_count(x):
    # 0: 0점, 1: 1점, 2: 2점, 3 이상: 3점
    if x == 0:
        return 0
    elif x == 1:
        return 1
    elif x == 2:
        return 2
    else:
        return 3

def score_child_ratio(x):
    # ~0.03: 1점, 0.03~0.06: 2점, 0.06~0.09: 3점, 0.09~0.12: 4점, 0.12~: 5점
    if x <= 0.03:
        return 1
    elif x <= 0.06:
        return 2
    elif x <= 0.09:
        return 3
    elif x <= 0.12:
        return 4
    else:
        return 5

def score_children_care_count(x):
    # 0~2: 1점, 3~5: 2점, 6~8: 3점, 9~11: 4점, 12~14: 5점, 15~17: 6점, 18~20: 7점, 21~23: 8점, 24~: 9점
    return 1 + (x // 3)

def score_high_diff(up, down):
    # 둘 다 40 이하: 4점, 둘 중 하나만 40 초과: 2점, 둘 다 40 초과: 0점
    if up <= 40 and down <= 40:
        return 4
    elif up > 40 and down > 40:
        return 0
    else:
        return 2

def score_residences_count(x):
    # 0~1999: 1점, 2000~3999: 2점, ... 18000 이상: 10점
    return 1 + (x // 2000)

# 각 점수 컬럼 생성
merged_candidate_df['score_bus'] = merged_candidate_df['bus_count'].apply(score_bus_count)
merged_candidate_df['score_train'] = merged_candidate_df['train_count'].apply(score_train_count)
merged_candidate_df['score_parking'] = merged_candidate_df['parking_count'].apply(score_parking_count)
merged_candidate_df['score_child_ratio'] = merged_candidate_df['어린이비율'].apply(score_child_ratio)
merged_candidate_df['score_children_care'] = merged_candidate_df['children_care_count'].apply(score_children_care_count)
merged_candidate_df['score_high'] = merged_candidate_df.apply(lambda row: score_high_diff(row['high_up'], row['high_down']), axis=1)
merged_candidate_df['score_residences'] = merged_candidate_df['residences_count'].apply(score_residences_count)

# 총점 컬럼 추가
score_cols = [
    'score_bus', 'score_train', 'score_parking', 'score_child_ratio',
    'score_children_care', 'score_high', 'score_residences'
]
merged_candidate_df['total_score'] = merged_candidate_df[score_cols].sum(axis=1)

merged_candidate_df.to_csv('../../Data_list/candidate_result/total_score.csv')

In [8]:
# 결과 확인
merged_candidate_df.head(2)

,cluster_id,data_point,시군구,어린이비율,contour,contour_mi,contour_ma,high_up,high_down,geometry,...,parking_count,children_care_count,score_bus,score_train,score_parking,score_child_ratio,score_children_care,score_high,score_residences,total_score
0,0,1976,부산광역시 금정구,0.061106,40.0,15.0,130.0,25.0,90.0,POINT (129.1003 35.21535),...,0,16,1.5,0,0,3,6,2,7,19.5
1,1,1883,부산광역시 해운대구,0.084278,55.0,20.0,210.0,35.0,155.0,POINT (129.15086 35.22258),...,0,7,2.0,0,0,3,3,2,3,13.0


In [14]:
merged_candidate_df.describe()

,cluster_id,data_point,어린이비율,contour,contour_mi,contour_ma,high_up,high_down,residences_count,bus_count,...,parking_count,children_care_count,score_bus,score_train,score_parking,score_child_ratio,score_children_care,score_high,score_residences,total_score
count,1140.000000,1140.000000,1139.000000,1140.000000,1123.000000,1123.000000,1123.000000,1123.000000,1140.000000,1140.000000,...,1140.000000,1140.000000,1140.000000,1140.000000,1140.000000,1140.000000,1140.000000,1140.000000,1140.000000,1140.000000
mean,569.500000,154.673684,0.102829,42.342105,21.424755,128.499555,20.017809,87.056990,3613.900000,3.058772,...,0.394737,3.121053,1.197807,0.032456,0.373684,3.744737,1.849123,2.307018,2.421053,11.925877
std,329.233959,262.580034,0.035921,55.400185,37.794835,103.276737,24.261823,65.758283,4270.311409,3.222924,...,0.834068,4.156433,0.788217,0.182171,0.718995,0.902011,1.305102,1.282195,2.063499,3.098045
min,0.000000,1.000000,0.029072,0.000000,-5.000000,0.000000,0.000000,0.000000,2.000000,0.000000,...,0.000000,0.000000,0.500000,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000,5.500000
25%,284.750000,7.000000,0.071018,5.000000,0.000000,45.000000,0.000000,25.000000,521.000000,0.000000,...,0.000000,0.000000,0.500000,0.000000,0.000000,3.000000,1.000000,2.000000,1.000000,9.500000
50%,569.500000,36.000000,0.113239,25.000000,5.000000,120.000000,10.000000,85.000000,1079.500000,2.000000,...,0.000000,1.000000,1.000000,0.000000,0.000000,4.000000,1.000000,2.000000,1.000000,11.500000
75%,854.250000,194.000000,0.154658,60.000000,25.000000,185.000000,30.000000,135.000000,5955.500000,5.000000,...,1.000000,5.000000,1.500000,0.000000,1.000000,5.000000,2.000000,4.000000,3.000000,14.000000
max,1139.000000,2419.000000,0.154658,405.000000,315.000000,545.000000,150.000000,275.000000,16985.000000,18.000000,...,8.000000,24.000000,5.000000,2.000000,3.000000,5.000000,9.000000,4.000000,9.000000,24.000000


In [10]:
# total_score 기준으로 내림차순 정렬 후 상위 100개 추출
top100_df = merged_candidate_df.sort_values(by='total_score', ascending=False).head(100)

# 결과 확인
top100_df.head(5)

,cluster_id,data_point,시군구,어린이비율,contour,contour_mi,contour_ma,high_up,high_down,geometry,...,parking_count,children_care_count,score_bus,score_train,score_parking,score_child_ratio,score_children_care,score_high,score_residences,total_score
40,40,902,부산광역시 동래구,0.094636,15.0,5.0,40.0,10.0,25.0,POINT (129.10381 35.203),...,0,17,2.0,0,0,4,6,4,8,24.0
38,38,1051,부산광역시 동래구,0.094636,10.0,5.0,45.0,5.0,35.0,POINT (129.10212 35.20053),...,0,18,2.0,0,0,4,7,4,7,24.0
8,8,1136,부산광역시 연제구,0.084428,70.0,40.0,235.0,30.0,165.0,POINT (129.096 35.17197),...,4,8,4.5,1,3,3,3,2,7,23.5
51,51,671,부산광역시 서구,0.061472,50.0,15.0,225.0,35.0,175.0,POINT (129.01542 35.10028),...,1,6,5.0,0,1,3,3,2,9,23.0
12,12,1101,부산광역시 동구,0.057847,40.0,5.0,215.0,35.0,175.0,POINT (129.04509 35.13414),...,2,15,3.5,0,2,2,6,2,6,21.5


In [11]:
top100_df.to_file('../../Data_list/candidate_result/top100.shp', encoding='cp949')

C:\Users\tjral\AppData\Local\Temp\ipykernel_32064\28838814.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  top100_df.to_file('../../Data_list/candidate_result/top100.shp', encoding='cp949')
c:\Users\tjral\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'residences_count' to 'residences'
  ogr_write(
c:\Users\tjral\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'train_count' to 'train_coun'
  ogr_write(
c:\Users\tjral\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'parking_count' to 'parking_co'
  ogr_write(
c:\Users\tjral\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'children_care_count' to 'children_c'
  ogr_write(
c:\Users\tjral\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'sc

In [15]:
들락날락 = gpd.read_file('../../Data_list/preprocessing_result/들락날락_등고수치_인구병합결과/위치_등고수치_시군구_어린이비율.shp')
들락날락.head(1)

,이름,주소,시군구,어린이비율,contour,contour_mi,contour_ma,high_up,high_down,geometry
0,다대도서관 들락날락,부산시 사하구 다대낙조2길 9,부산광역시 사하구,0.068876,40.0,0.0,100.0,40.0,60.0,POINT (128.9647 35.05043)


In [17]:
recent_gdf = pd.read_csv('../../Data_list/preprocessing_result/스코어링할데이터/recent_gdf.csv')
recent_gdf.head(1)

,주소번호,이름,주소,위도,경도,geometry,residences_count,bus_count,train_count,parking_count,children_care_count,어린이,high_up,high_down
0,27,다대도서관 들락날락,부산시 사하구 다대낙조2길 9,35.050427,128.9647,POINT (128.964700408 35.05042701),898,9,0,1,5,0.068876,40.0,60.0


In [ ]:
# 중복 컬럼을 제외한 recent_gdf의 컬럼만 추출
duplicate_cols = set(들락날락.columns) & set(recent_gdf.columns)
recent_gdf_unique = recent_gdf[[col for col in recent_gdf.columns if col not in duplicate_cols]]

# 들락날락과 recent_gdf_unique를 컬럼 기준으로 합침 (index 맞추기)
merged_recent_df = pd.concat([들락날락.reset_index(drop=True), recent_gdf_unique.reset_index(drop=True)], axis=1)

In [ ]:
# 중복되는 데이터가 들어간 컬럼 제거
merged_recent_df = merged_recent_df.drop(columns=['어린이', '위도', '경도'])
merged_recent_df.head(2)

,이름,주소,시군구,어린이비율,contour,contour_mi,contour_ma,high_up,high_down,geometry,주소번호,residences_count,bus_count,train_count,parking_count,children_care_count
0,다대도서관 들락날락,부산시 사하구 다대낙조2길 9,부산광역시 사하구,0.068876,40.0,0.0,100.0,40.0,60.0,POINT (128.9647 35.05043),27,898,9,0,1,5
1,더나눔어린이작은도서관 들락날락,"동구 영초윗길 48(초량동), 장기려기념관 1층",부산광역시 동구,0.057847,60.0,10.0,215.0,50.0,155.0,POINT (129.03265 35.11853),28,12521,4,0,3,7


In [22]:
# 일부 object 형태의 컬럼의 데이터타입 모두 숫자로 변경
merged_recent_df[['contour_mi', 'contour_ma', 'high_up', 'high_down']] = merged_recent_df[['contour_mi', 'contour_ma', 'high_up', 'high_down']].astype('float64')
merged_recent_df.dtypes

이름                       object
주소                       object
시군구                      object
어린이비율                   float64
contour                 float64
contour_mi              float64
contour_ma              float64
high_up                 float64
high_down               float64
geometry               geometry
주소번호                      int64
residences_count          int64
bus_count                 int64
train_count               int64
parking_count             int64
children_care_count       int64
score_bus               float64
score_train               int64
score_parking             int64
score_child_ratio         int64
score_children_care       int64
dtype: object

In [23]:
# 각 점수 컬럼 생성
merged_recent_df['score_bus'] = merged_recent_df['bus_count'].apply(score_bus_count)
merged_recent_df['score_train'] = merged_recent_df['train_count'].apply(score_train_count)
merged_recent_df['score_parking'] = merged_recent_df['parking_count'].apply(score_parking_count)
merged_recent_df['score_child_ratio'] = merged_recent_df['어린이비율'].apply(score_child_ratio)
merged_recent_df['score_children_care'] = merged_recent_df['children_care_count'].apply(score_children_care_count)
merged_recent_df['score_high'] = merged_recent_df.apply(lambda row: score_high_diff(row['high_up'], row['high_down']), axis=1)
merged_recent_df['score_residences'] = merged_recent_df['residences_count'].apply(score_residences_count)

# 총점 컬럼 추가
score_cols = [
    'score_bus', 'score_train', 'score_parking', 'score_child_ratio',
    'score_children_care', 'score_high', 'score_residences'
]
merged_recent_df['total_score'] = merged_recent_df[score_cols].sum(axis=1)

merged_recent_df.to_csv('../../Data_list/recent_result/total_score.csv')

In [24]:
merged_recent_df.head(1)

,이름,주소,시군구,어린이비율,contour,contour_mi,contour_ma,high_up,high_down,geometry,...,parking_count,children_care_count,score_bus,score_train,score_parking,score_child_ratio,score_children_care,score_high,score_residences,total_score
0,다대도서관 들락날락,부산시 사하구 다대낙조2길 9,부산광역시 사하구,0.068876,40.0,0.0,100.0,40.0,60.0,POINT (128.9647 35.05043),...,1,5,2.5,0,1,3,2,2,1,11.5


In [26]:
merged_recent_df.describe()

,어린이비율,contour,contour_mi,contour_ma,high_up,high_down,주소번호,residences_count,bus_count,train_count,parking_count,children_care_count,score_bus,score_train,score_parking,score_child_ratio,score_children_care,score_high,score_residences,total_score
count,106.000000,106.000000,106.000000,106.000000,106.000000,106.000000,106.000000,106.000000,106.000000,106.000000,106.000000,106.000000,106.000000,106.000000,106.000000,106.000000,106.000000,106.000000,106.000000,106.000000
mean,0.073811,40.943396,18.349057,145.141509,22.594340,104.198113,121.537736,6847.622642,5.226415,0.103774,0.367925,5.301887,1.688679,0.103774,0.367925,2.990566,2.462264,2.037736,3.971698,13.622642
std,0.022411,53.242988,32.202645,97.912637,27.231207,61.691000,74.947115,4602.087428,3.737285,0.306415,0.708123,4.234034,0.908827,0.306415,0.708123,0.696868,1.395056,1.202573,2.323616,3.519883
min,0.029072,0.000000,0.000000,5.000000,0.000000,0.000000,27.000000,185.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000,5.500000
25%,0.060624,5.000000,5.000000,80.000000,5.000000,60.000000,57.250000,3262.500000,2.250000,0.000000,0.000000,2.000000,1.000000,0.000000,0.000000,3.000000,1.000000,2.000000,2.000000,11.000000
50%,0.068876,22.500000,5.000000,140.000000,10.000000,102.500000,88.500000,5901.000000,5.000000,0.000000,0.000000,5.000000,1.500000,0.000000,0.000000,3.000000,2.000000,2.000000,3.500000,14.000000
75%,0.084278,50.000000,20.000000,198.750000,30.000000,145.000000,205.750000,10603.750000,7.000000,0.000000,1.000000,7.750000,2.000000,0.000000,1.000000,3.000000,3.000000,2.000000,6.000000,16.000000
max,0.154658,300.000000,250.000000,495.000000,155.000000,305.000000,236.000000,17063.000000,17.000000,1.000000,3.000000,19.000000,4.500000,1.000000,3.000000,5.000000,7.000000,4.000000,9.000000,22.500000
